# 🏆 Gold - Classificação Brasileirão 2026

## Descrição
Este notebook agrega os dados da camada Silver para métricas de negócio.

### Pipeline ETL
- **Extract**: Lê do Parquet Silver
- **Transform**: Agregações, métricas, rankings
- **Load**: Salva em `Files/gold/classificacao.parquet`

In [ ]:
# Import libraries
import pandas as pd
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Extract - Read from Silver
logger.info("Lendo dados da camada Silver...")
df = pd.read_parquet("Files/silver/classificacao.parquet")
logger.info(f"Dados lidos: {len(df)} registros")
df.head()

In [ ]:
# Transform - Agregações e métricas de negócio
logger.info("Calculando métricas de negócio...")

df_gold = df.copy()

# 1. Adicionar zona de classificação
def get_zona(posicao):
    if posicao <= 4:
        return "libertadores"
    elif posicao <= 6:
        return "libertadores_pre"
    elif posicao <= 12:
        return "sul_americana"
    elif posicao <= 16:
        return "serie_a"
    else:
        return "rebaixamento"

df_gold["zona"] = df_gold["posição"].apply(get_zona)

# 2. Calcular eficiência (pontos por jogo)
df_gold["eficiencia"] = (df_gold["pts"] / df_gold["j"]).round(2)

# 3. Calcular média de golos
df_gold["media_golos_marcados"] = (df_gold["gp"] / df_gold["j"]).round(2)
df_gold["media_golos_sofridos"] = (df_gold["gc"] / df_gold["j"]).round(2)

# 4. Adicionar metadata
from datetime import datetime
df_gold["processed_at"] = datetime.now().isoformat()
df_gold["camada"] = "gold"

logger.info(f"Métricas calculadas: {len(df_gold)} registros")
df_gold.head()

In [ ]:
# Load - Save to Gold Parquet
output_path = "Files/gold/classificacao.parquet"
import os
os.makedirs("Files/gold", exist_ok=True)
df_gold.to_parquet(output_path, index=False)
logger.info(f"Dados salvos em: {output_path}")

# Display final result
df_gold